# Sesion 8 — De Datos Crudos a Dataset Analizable
## Diplomado: Machine Learning en Seguros · FC UNAM
### 2 de mayo de 2026  ·  07:00 - 11:00 h  (4 horas)

---

> **Premisa de la sesion:** recibes los archivos crudos de la aseguradora.
> Tu trabajo: convertirlos en un dataset limpio, bien tipado y eficiente,
> resolviendo 7 problemas reales uno por uno.

---

**Prerequisito:** Debe existir la carpeta `datos/` con los archivos CSV.

## Las 7 Dudas que Resolvemos Hoy

| # | Duda | Herramienta |
|---|------|-------------|
| 1 | 46 columnas — ¿cuales necesito? | Taxonomia + `usecols` |
| 2 | Texto sucio: M/F/Masculino/Femenino | `str` operations |
| 3 | Fechas como texto: '15/04/2026' | `pd.to_datetime()` |
| 4 | API de reaseguro devuelve JSON | `read_json()` + `json_normalize()` |
| 5 | 90k filas, 11MB sin esperar | `chunks` + `category` |
| 6 | Downcast rompio precision de primas | Estrategia segura de `dtypes` |
| 7 | ¿Cuando cambiar a Polars? | `polars` — intro y comparativa |

---
## ACT 1 — El Dataset Crudo

### Duda 1: 46 columnas — ¿cuales necesito?

In [2]:
import pandas as pd
import numpy as np
import os, time


# ── Paso 1: Cargar TODO para entender que hay ────────────────────────────────
# Primera regla: antes de descartar, entende lo que tienes
df_todo = pd.read_csv('datos/cartera_polizas.csv', nrows=5)  # solo 5 filas para ver
print(f'Columnas totales: {len(df_todo.columns)}')
print()
print('Lista de columnas:')
for i, col in enumerate(df_todo.columns, 1):
    print(f'  {i:>2}. {col}')

Columnas totales: 46

Lista de columnas:
   1. id_poliza
   2. num_poliza
   3. id_contrato_interno
   4. folio_emision
   5. id_sistema_legacy
   6. nombre
   7. apellido_paterno
   8. apellido_materno
   9. nombre_completo
  10. rfc
  11. fecha_nacimiento
  12. edad
  13. sexo
  14. estado_civil
  15. ocupacion
  16. nivel_educacion
  17. ramo
  18. plan
  19. fecha_emision
  20. fecha_inicio_vigencia
  21. fecha_fin_vigencia
  22. num_renovaciones
  23. status_poliza
  24. motivo_baja
  25. canal_venta
  26. marca_vehiculo
  27. modelo_vehiculo
  28. tipo_vehiculo
  29. suma_asegurada
  30. deducible
  31. prima_neta
  32. prima_total
  33. cuota_prima
  34. forma_pago
  35. num_cuotas
  36. agente_id
  37. estado
  38. municipio
  39. codigo_postal
  40. coord_lat
  41. coord_lon
  42. version_documento
  43. hash_documento
  44. timestamp_carga
  45. usuario_captura
  46. ip_carga


In [3]:
# ── Paso 2: Diagnostico de todas las columnas ────────────────────────────────
# Cargamos todo para el diagnostico inicial
df_full = pd.read_csv('datos/cartera_polizas.csv')
mb_full = df_full.memory_usage(deep=True).sum() / 1024**2
print(f'Dataset completo: {df_full.shape} · {mb_full:.1f} MB')
print()

# Perfil de cada columna: tipo, NaN%, valores unicos
print(f'{"Columna":<30} {"Dtype":<12} {"NaN%":>7} {"Unicos":>8}')
print('-' * 65)
for col in df_full.columns:
    dtype = str(df_full[col].dtype)
    nan_pct = df_full[col].isna().mean() * 100
    n_uniq  = df_full[col].nunique()
    print(f'{col:<30} {dtype:<12} {nan_pct:>6.1f}% {n_uniq:>8,}')

Dataset completo: (50000, 46) · 112.0 MB

Columna                        Dtype           NaN%   Unicos
-----------------------------------------------------------------
id_poliza                      object          0.0%   50,000
num_poliza                     object          0.0%   50,000
id_contrato_interno            object          0.0%   48,572
folio_emision                  object          0.0%   49,870
id_sistema_legacy              object          0.0%   49,988
nombre                         object          0.0%       55
apellido_paterno               object          0.0%       38
apellido_materno               object          0.0%       23
nombre_completo                object          0.0%   31,140
rfc                            object          0.0%   50,000
fecha_nacimiento               object          0.0%   15,676
edad                           int64           0.0%       46
sexo                           object          0.0%        4
estado_civil                   object 

In [4]:
# ── Paso 3: Clasificar las columnas por categoria ────────────────────────────
# Esta clasificacion es una DECISION DE NEGOCIO — no solo tecnica

ANALITICAS = [
    'id_poliza','num_poliza','ramo','plan','status_poliza',
    'nombre','apellido_paterno','apellido_materno',
    'rfc','edad','sexo','estado_civil','ocupacion',
    'fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia',
    'num_renovaciones','motivo_baja',
    'suma_asegurada','deducible','prima_neta','prima_total',
    'forma_pago','agente_id','canal_venta',
    'estado','municipio','codigo_postal',
    'marca_vehiculo','modelo_vehiculo','tipo_vehiculo','fecha_nacimiento',  # solo Autos
]

REDUNDANTES = [
    'nombre_completo',        # = nombre + ap_pat + ap_mat
    'id_contrato_interno',    # ≈ id_poliza
    'folio_emision',          # ≈ num_poliza
    'cuota_prima',            # = prima_total / num_cuotas
    'num_cuotas',             # derivado de forma_pago
]

ADMINISTRATIVAS = [
    'hash_documento',         # hash SHA del PDF — auditoria IT
    'timestamp_carga',        # mismo valor para todos
    'ip_carga',               # IP del servidor batch
    'usuario_captura',        # operacion interna
    'version_documento',      # version del formato del contrato
    'id_sistema_legacy',      # util SOLO para joins con sistema core
]

CONDICIONALES = [
    'nivel_educacion',        # si lo necesitas para el modelo
    'coord_lat','coord_lon',  # solo si haces analisis geoespacial
]

print(f'Analiticas:     {len(ANALITICAS)}')
print(f'Redundantes:    {len(REDUNDANTES)}')
print(f'Administrativas:{len(ADMINISTRATIVAS)}')
print(f'Condicionales:  {len(CONDICIONALES)}')
print(f'Total:          {len(ANALITICAS)+len(REDUNDANTES)+len(ADMINISTRATIVAS)+len(CONDICIONALES)}')

Analiticas:     32
Redundantes:    5
Administrativas:6
Condicionales:  3
Total:          46


In [ ]:
# ── Paso 4: Cargar SOLO las columnas que necesitamos ─────────────────────────
t0 = time.time()
df_final = pd.read_csv(
    'datos/cartera_polizas.csv',
    usecols=ANALITICAS,
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)
t1 = time.time()
mb_opt = df_final.memory_usage(deep=True).sum() / 1024**2

print('=== COMPARATIVA DE CARGA ===')
print(f'Carga completa (46 cols):  {mb_full:.1f} MB')
print(f'Carga selectiva ({len(ANALITICAS)} cols):  {mb_opt:.1f} MB')
print(f'Reduccion de memoria:      {(1-mb_opt/mb_full)*100:.0f}%')
print(f'Tiempo de carga:           {(t1-t0)*1000:.0f} ms')
print()
print(f'Dataset de trabajo: {df_final.shape}')
df_final.head(3)

=== COMPARATIVA DE CARGA ===
Carga completa (46 cols):  112.0 MB
Carga selectiva (32 cols):  73.7 MB
Reduccion de memoria:      34%
Tiempo de carga:           258 ms

Dataset de trabajo: (50000, 32)


,id_poliza,num_poliza,nombre,apellido_paterno,apellido_materno,rfc,fecha_nacimiento,edad,sexo,estado_civil,...,tipo_vehiculo,suma_asegurada,deducible,prima_neta,prima_total,forma_pago,agente_id,estado,municipio,codigo_postal
0,POL-000001,Vid-21-000001,Gabriela,Moreno,Vega,MOGV020429CG6,29/04/2002,24,F,Union libre,...,NaN,3000000,NaN,54000.0,67651.20,Mensual,AG054,Veracruz,Poza Rica,36619.0
1,POL-000002,Aut-19-000002,Valeria,Torres,Castillo,TOVC020815IA8,15/08/2002,23,Femenino,Casado,...,Compacto,150000,8000.0,5250.0,6394.50,Trimestral,AG004,Michoacan,Morelia,58889.0
2,POL-000003,GMM-22-000003,Fernanda,Ramos,Silva,RAFS941018BC1,18/10/1994,31,M,Union libre,...,NaN,800000,5000.0,17600.0,22049.28,Mensual,AG051,Baja California,Tecate,45784.0


### 📝 Ejercicio 1 — Auditoria de columnas (8 min)

Usando el perfil que generaste arriba:
- **1a.** Identifica las 3 columnas con mas NaN — ¿tienen sentido esos NaN o son errores?
- **1b.** Encuentra columnas con menos de 5 valores unicos — ¿cuales deberian ser `category`?
- **1c.** Hay una columna numerica con un rango imposible (negativo o muy alto) — ¿cual es?
  *Pista: usa `df.describe()` sobre las columnas analiticas*

In [ ]:
# Tu codigo aqui:
Columnas_NAN = df_final.isna().sum().sort_values(ascending=False)
print(Columnas_NAN.head(3))
print()
print(df_final.groupby('ramo')['id_poliza'].count())
print()
Columnas_unicos = df_final.nunique().sort_values()
print(Columnas_unicos[Columnas_unicos < 5])
print()
print(df_final.describe().round(2))

motivo_baja       47535
marca_vehiculo    35248
tipo_vehiculo     35248
dtype: int64

ramo
Accidentes Personales     5207
Autos                    14752
GMM                      22531
Vida                      7510
Name: id_poliza, dtype: int64

ramo             4
sexo             4
motivo_baja      4
status_poliza    4
forma_pago       4
dtype: int64

           edad  num_renovaciones  modelo_vehiculo  suma_asegurada  deducible  \
count  50000.00          50000.00         14752.00        50000.00   42490.00   
mean      43.32              1.24          2020.00      1002000.00    8266.82   
std       12.96              1.13             3.17      1105162.53    5503.03   
min       21.00              0.00          2015.00       150000.00    1000.00   
25%       32.00              0.00          2017.00       300000.00    5000.00   
50%       43.00              1.00          2020.00       500000.00    8000.00   
75%       54.00              2.00          2023.00      1000000.00   10000.00 

---
### Duda 2: Texto Sucio — str Operations

El campo `sexo` tiene 4 representaciones distintas del mismo valor.
Si no lo normalizas, `groupby('sexo')` produce 8 grupos en lugar de 2.

In [ ]:
# ── Ver el problema ──────────────────────────────────────────────────────────
print('Valores unicos en sexo ANTES de limpiar:')
print(df_final['sexo'].value_counts(dropna=False))
print(f'Total valores distintos: {df_final["sexo"].nunique()}')
print()

# Si haces groupby ahora, obtienes grupos incorrectos:
grupos_incorrectos = df_final.groupby('sexo')['prima_total'].mean()
print(f'Grupos sin limpiar: {len(grupos_incorrectos)} (deberian ser 2)')

Valores unicos en sexo ANTES de limpiar:
sexo
M            16697
F            16585
Femenino      8466
Masculino     8252
Name: count, dtype: int64
Total valores distintos: 4

Grupos sin limpiar: 4 (deberian ser 2)


In [ ]:
# ── Solucion: normalizar con str operations ──────────────────────────────────

# Paso 1: normalizar a mayusculas sin espacios
df_final['sexo'] = df_final['sexo'].str.strip().str.upper()

# Paso 2: mapear todas las variantes al estandar
MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
}
df_final['sexo'] = df_final['sexo'].map(MAPA_SEXO)
# Valores no reconocidos quedan como NaN automaticamente

print('DESPUES de normalizar:')
print(df_final['sexo'].value_counts(dropna=False))
print()
# Ahora groupby correcto
print('Prima promedio por sexo:')
print(df_final.groupby('sexo')['prima_total'].mean().round(2))

DESPUES de normalizar:
sexo
F    25051
M    24949
Name: count, dtype: int64

Prima promedio por sexo:
sexo
F    29138.45
M    29428.49
Name: prima_total, dtype: float64


In [ ]:
# ── Mas str operations sobre los datos reales ────────────────────────────────

# Limpiar codigo_postal: eliminar 'N/D' (ya convertido a NaN por na_values)
# Verificar:
print(f'CP con NaN: {df_final["codigo_postal"].isna().sum()}')
# Rellenar CP desconocido con 'DESCONOCIDO' para no perder la fila
df_final['codigo_postal'] = df_final['codigo_postal'].fillna('DESCONOCIDO')

# Extraer ramo y anio desde num_poliza ('GMM-24-000123')
df_final['ramo_codigo']  = df_final['num_poliza'].str.extract(r'^([A-Za-z]+)-')
df_final['anio_poliza']  = df_final['num_poliza'].str.extract(r'-([0-9]{2})-').astype(float).astype('Int16')

# Verificar
print(df_final[['num_poliza','ramo_codigo','anio_poliza']].head(5).to_string(index=False))

CP con NaN: 1500
   num_poliza ramo_codigo  anio_poliza
Vid-21-000001         Vid           21
Aut-19-000002         Aut           19
GMM-22-000003         GMM           22
Vid-19-000004         Vid           19
Vid-20-000005         Vid           20


---
### Duda 3: Fechas como Texto — pd.to_datetime()

In [ ]:
# ── El problema: fechas llegaron como strings ────────────────────────────────
print('Tipo ANTES de convertir:', df_final['fecha_nacimiento'].dtype)
print('Muestra:', df_final['fecha_nacimiento'].head(3).values)
print()

# ── Convertir fecha_nacimiento (formato d/m/Y) ────────────────────────────────
df_final['fecha_nacimiento'] = pd.to_datetime(
    df_final['fecha_nacimiento'],
    format='%d/%m/%Y',
    errors='coerce'  # fechas invalidas → NaT (no detiene el proceso)
)

# ── Convertir fechas ISO estandar ────────────────────────────────────────────
for col_fecha in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    df_final[col_fecha] = pd.to_datetime(df_final[col_fecha], errors='coerce')

print('Fechas convertidas:')
print(df_final[['fecha_nacimiento','fecha_emision','fecha_fin_vigencia']].dtypes)
print(f'NaT en fecha_nacimiento: {df_final["fecha_nacimiento"].isna().sum()}')

Tipo ANTES de convertir: object
Muestra: ['29/04/2002' '15/08/2002' '18/10/1994']

Fechas convertidas:
fecha_nacimiento      datetime64[ns]
fecha_emision         datetime64[ns]
fecha_fin_vigencia    datetime64[ns]
dtype: object
NaT en fecha_nacimiento: 0


In [ ]:
# ── Calculos actuariales con las fechas convertidas ──────────────────────────
hoy = pd.Timestamp.today()

# Edad calculada (mas precisa que la columna 'edad' del CSV)
df_final['edad_calc'] = ((hoy - df_final['fecha_nacimiento']).dt.days / 365.25)

# Dias de vigencia de la poliza
df_final['dias_vigencia'] = (df_final['fecha_fin_vigencia'] - df_final['fecha_inicio_vigencia']).dt.days

# Fraccion expuesta (cuanto del periodo ya transcurrio)
dias_transcurridos = (hoy - df_final['fecha_inicio_vigencia']).dt.days
df_final['fraccion_expuesta'] = (dias_transcurridos / df_final['dias_vigencia']).clip(0, 1).round(4)

# Componentes de fecha para groupby temporal
df_final['anio_emision']     = df_final['fecha_emision'].dt.year
df_final['mes_emision']      = df_final['fecha_emision'].dt.month
df_final['trimestre_emision']= df_final['fecha_emision'].dt.quarter

# Verificar
print(df_final[['nombre','edad','edad_calc','dias_vigencia','fraccion_expuesta']].head(5).to_string(index=False))

  nombre  edad  edad_calc  dias_vigencia  fraccion_expuesta
Gabriela    24  24.049281            365                1.0
 Valeria    23  23.753593            366                1.0
Fernanda    31  31.578371            365                1.0
  Silvia    40  40.279261            366                1.0
 Antonio    29  29.817933            365                1.0


### 📝 Ejercicio 2 — Limpiar siniestros.csv (10 min)

El archivo `siniestros.csv` tiene fechas en 3 formatos distintos:
- `fecha_ocurrencia`: YYYY-MM-DD ISO 8601
- `fecha_apertura`: d/m/Y (mismo dia que fecha_reporte pero formato distinto — es REDUNDANTE)
- `fecha_ultimo_movimiento`: d/m/Y

**2a.** Carga siniestros.csv usando SOLO las columnas utiles (descarta las administrativas y redundantes).

**2b.** Convierte las 3 columnas de fecha a datetime con el formato correcto.

**2c.** Calcula `dias_reporte_real` = fecha_reporte - fecha_ocurrencia.

**2d.** Calcula `dias_resolucion_real` = fecha_cierre - fecha_reporte (NaT si no esta cerrado).

**2e.** Verifica: ¿cuantos siniestros llevan mas de 180 dias sin cerrar?

In [12]:
# Tu codigo aqui:
siniestros_cols_utiles = [
    'id_siniestro','id_poliza','ramo','tipo_siniestro',
    'fecha_ocurrencia','fecha_reporte','fecha_ultimo_movimiento','fecha_cierre',
    'dias_reporte','monto_reclamado','monto_pagado',
    'status_siniestro','motivo_rechazo','id_ajustador',
]
# Carga, convierte fechas y calcula los campos derivados:
siniestros = pd.read_csv('datos/siniestros.csv', usecols = siniestros_cols_utiles)

siniestros['fecha_ocurrencia'] = pd.to_datetime(siniestros['fecha_ocurrencia'], format='%Y-%m-%d', errors='coerce')
siniestros['fecha_reporte'] = pd.to_datetime(siniestros['fecha_reporte'], format='%Y-%m-%d', errors='coerce')
siniestros['fecha_cierre'] = pd.to_datetime(siniestros['fecha_cierre'], format='%Y-%m-%d', errors='coerce')
siniestros['fecha_ultimo_movimiento'] = pd.to_datetime(siniestros['fecha_ultimo_movimiento'], format='%d/%m/%Y', errors='coerce')

n_nat = siniestros['fecha_ocurrencia'].isna().sum()
print(f'Nas en fecha_ocurrencia: {n_nat} ({n_nat/len(siniestros)*100:.1f}%)')

siniestros['dias_reporte'] = (siniestros['fecha_reporte'] - siniestros['fecha_ocurrencia']).dt.days


siniestros['días_resolucion'] = (siniestros['fecha_cierre'] - siniestros['fecha_reporte']).dt.days

print(siniestros[['id_siniestro','fecha_ocurrencia','fecha_reporte','dias_reporte','días_resolucion']].head(5).to_string(index=False))

print(siniestros[siniestros['días_resolucion'] > 180])





Nas en fecha_ocurrencia: 0 (0.0%)
id_siniestro fecha_ocurrencia fecha_reporte  dias_reporte  días_resolucion
 SIN-0000001       2022-06-22    2022-06-25             3              NaN
 SIN-0000002       2023-04-07    2023-04-14             7              NaN
 SIN-0000003       2022-05-24    2022-05-26             2              NaN
 SIN-0000004       2021-10-29    2021-10-30             1              NaN
 SIN-0000005       2023-11-15    2023-11-18             3            180.0
      id_siniestro   id_poliza fecha_ocurrencia fecha_reporte  \
5      SIN-0000006  POL-027460       2024-12-09    2024-12-10   
11     SIN-0000012  POL-000317       2024-08-02    2024-08-02   
94     SIN-0000095  POL-044901       2021-08-03    2021-08-03   
120    SIN-0000121  POL-006042       2021-09-06    2021-09-11   
127    SIN-0000128  POL-007437       2020-03-29    2020-04-05   
...            ...         ...              ...           ...   
17945  SIN-0017946  POL-011131       2024-06-04    2024-06-04

---
### Duda 4: JSON — Datos de API


In [13]:
# ── Simular una respuesta JSON de una API de reaseguro ───────────────────────
import json

# Este es el tipo de JSON que recibirias de una API REST
respuesta_api = {
    'timestamp': '2026-05-02T07:00:00',
    'origen': 'Sistema_Reaseguro_v3.1',
    'polizas_reaseguradas': [
        {'id_poliza':'POL-000001','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':960.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
        {'id_poliza':'POL-000002','ramo':'Vida',
         'reasegurador':{'nombre':'Swiss Re','participacion':0.35,'prima':1820.0},
         'limites':{'maximo':5_000_000,'retencion':1_000_000}},
        {'id_poliza':'POL-000005','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':550.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
    ]
}

# Guardar como JSON (simula lo que llegaria de la API)
with open('datos/respuesta_reaseguro.json','w',encoding='utf-8') as f:
    json.dump(respuesta_api, f, ensure_ascii=False, indent=2)

print('JSON guardado en datos/respuesta_reaseguro.json')
print('Primeras lineas:')
print(json.dumps(respuesta_api, indent=2, ensure_ascii=False)[:300])

JSON guardado en datos/respuesta_reaseguro.json
Primeras lineas:
{
  "timestamp": "2026-05-02T07:00:00",
  "origen": "Sistema_Reaseguro_v3.1",
  "polizas_reaseguradas": [
    {
      "id_poliza": "POL-000001",
      "ramo": "GMM",
      "reasegurador": {
        "nombre": "Munich Re",
        "participacion": 0.4,
        "prima": 960.0
      },
      "limites": 


In [14]:
# ── Problema: JSON anidado no se carga directo en DataFrame ─────────────────
# pd.read_json funciona para JSON simple pero no para estructuras anidadas

# Cargar el JSON
with open('datos/respuesta_reaseguro.json') as f:
    data = json.load(f)

# Intentar con pd.read_json — no da el resultado esperado con anidados
# pd.read_json('datos/respuesta_reaseguro.json')  # solo lee nivel 1

# ── Solucion: json_normalize aplana el JSON anidado ──────────────────────────
from pandas import json_normalize

df_reas = json_normalize(
    data['polizas_reaseguradas'],
    sep='_'    # separador para campos anidados
)

print('DataFrame aplanado:')
print(df_reas.to_string(index=False))
print()
print('Columnas generadas:', list(df_reas.columns))

DataFrame aplanado:
 id_poliza ramo reasegurador_nombre  reasegurador_participacion  reasegurador_prima  limites_maximo  limites_retencion
POL-000001  GMM           Munich Re                        0.40               960.0         2000000             500000
POL-000002 Vida            Swiss Re                        0.35              1820.0         5000000            1000000
POL-000005  GMM           Munich Re                        0.40               550.0         2000000             500000

Columnas generadas: ['id_poliza', 'ramo', 'reasegurador_nombre', 'reasegurador_participacion', 'reasegurador_prima', 'limites_maximo', 'limites_retencion']


In [15]:
# ── json_normalize con listas anidadas ───────────────────────────────────────
# Caso mas complejo: cuando hay listas dentro del JSON

respuesta_multi = {
    'polizas': [
        {'id':'P01','coberturas':[{'tipo':'Hospitalizacion','suma':500_000},{'tipo':'Cirugia','suma':300_000}]},
        {'id':'P02','coberturas':[{'tipo':'Hospitalizacion','suma':800_000}]},
    ]
}

# record_path: donde esta la lista a 'explotar'
# meta: campos del nivel padre que quieres conservar
df_cob = json_normalize(
    respuesta_multi['polizas'],
    record_path='coberturas',
    meta=['id'],
    sep='_'
)
print(df_cob)

              tipo    suma   id
0  Hospitalizacion  500000  P01
1          Cirugia  300000  P01
2  Hospitalizacion  800000  P02


In [ ]:
# ── Guardar DataFrame como JSON ──────────────────────────────────────────────

# orient='records' — lista de objetos (lo mas comun para APIs)
df_final.head(10).to_json('datos/muestra.json',
    orient='records',
    force_ascii=False,  # preserva caracteres especiales (acentos)
    indent=2,
    date_format='iso'   # fechas en formato ISO
)

# Verificar
with open('datos/muestra.json') as f:
    preview = f.read(300)
print(preview)

[
  {
    "id_poliza":"POL-000001",
    "num_poliza":"Vid-21-000001",
    "nombre":"Gabriela",
    "apellido_paterno":"Moreno",
    "apellido_materno":"Vega",
    "rfc":"MOGV020429CG6",
    "fecha_nacimiento":"2002-04-29T00:00:00.000",
    "edad":24,
    "sexo":"F",
    "estado_civil":"Union libre",


---
### Duda 5: 90k Filas — Procesar sin Esperar (chunks)

In [17]:
# ── Ver el tamano del archivo de beneficiarios ───────────────────────────────
mb_ben = os.path.getsize('datos/beneficiarios.csv') / 1024**2
print(f'beneficiarios.csv: {mb_ben:.1f} MB')

# Contar filas sin cargar todo
total_ben = sum(len(chunk) for chunk in
    pd.read_csv('datos/beneficiarios.csv', chunksize=10_000))
print(f'Total beneficiarios: {total_ben:,}')

# ── Patron real: calcular estadisticas por parentesco ─────────────────────────
# Sin cargar los 90k en memoria a la vez
conteo_parentesco = {}

for chunk in pd.read_csv('datos/beneficiarios.csv', chunksize=10_000):
    counts = chunk['parentesco'].value_counts().to_dict()
    for k, v in counts.items():
        conteo_parentesco[k] = conteo_parentesco.get(k, 0) + v

resultado = pd.Series(conteo_parentesco).sort_values(ascending=False)
print('Beneficiarios por parentesco (procesado por chunks):')
print(resultado)

beneficiarios.csv: 11.5 MB
Total beneficiarios: 90,000
Beneficiarios por parentesco (procesado por chunks):
Conyuge    26718
Hijo       22785
Hija       17871
Madre       7298
Padre       7173
Hermano     3596
Hermana     2765
Otro        1794
dtype: int64


In [18]:
# ── Filtrar y concatenar solo lo que necesitas ───────────────────────────────
# Obtener solo beneficiarios activos de polizas de Vida

partes = []
for chunk in pd.read_csv('datos/beneficiarios.csv',
                         chunksize=10_000,
                         usecols=['id_beneficiario','id_poliza','nombre',
                                  'apellido_paterno','parentesco','porcentaje','activo']):
    filtrado = chunk[chunk['activo'] == True]
    if len(filtrado) > 0:
        partes.append(filtrado)

ben_activos = pd.concat(partes, ignore_index=True)
print(f'Beneficiarios activos: {len(ben_activos):,}')
print(f'Memoria: {ben_activos.memory_usage(deep=True).sum()/1024**2:.0f} KB')

Beneficiarios activos: 67,656
Memoria: 21 KB


---
### Duda 6: Downcast — Cuándo Es Seguro

In [19]:
# ── Demostrar el problema de precision ───────────────────────────────────────
import numpy as np

# float64 vs float32 con valores grandes
prima_grande = 15_432_756.80
print(f'Original (float64): {prima_grande}')
print(f'Como float32:       {np.float32(prima_grande)}')
print(f'Diferencia:         {abs(prima_grande - float(np.float32(prima_grande))):.2f}')
print()

# Con valores tipicos de primas individuales
prima_gmm = 3_450.75
print(f'Prima GMM (float64): {prima_gmm}')
print(f'Prima GMM (float32): {np.float32(prima_gmm)}')
print(f'Diferencia:          {abs(prima_gmm - float(np.float32(prima_gmm))):.6f}')

Original (float64): 15432756.8
Como float32:       15432757.0
Diferencia:         0.20

Prima GMM (float64): 3450.75
Prima GMM (float32): 3450.75
Diferencia:          0.000000


In [ ]:
# ── Estrategia segura de optimizacion ────────────────────────────────────────
print('Memoria ANTES de optimizar:')
print(f'{df_final.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print()

df_opt = df_final.copy()

# SEGURO: categoricas para columnas con pocos valores unicos
cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                 'forma_pago','estado','estado_civil','tipo_vehiculo']
for col in cols_category:
    if col in df_opt.columns:
        n_uniq = df_opt[col].nunique()
        n_tot  = len(df_opt)
        pct = n_uniq/n_tot
        print(f'  {col:<25}: {n_uniq:>5} unicos ({pct:.1%}) → category')
        df_opt[col] = df_opt[col].astype('category')

# SEGURO: enteros pequenos
df_opt['num_renovaciones'] = df_opt['num_renovaciones'].fillna(0).astype('int8')

# SEGURO: booleano
# (activa ya no esta porque la descartamos, pero aplica el principio)

# CONSERVAR en float64: primas y sumas (montos grandes o con centavos importantes)
# NO hacer: df_opt['prima_total'] = df_opt['prima_total'].astype('float32')

print()
print('Memoria DESPUES de optimizar:')
mb_antes = df_final.memory_usage(deep=True).sum()/1024**2
mb_desp  = df_opt.memory_usage(deep=True).sum()/1024**2
print(f'{mb_antes:.2f} MB → {mb_desp:.2f} MB ({(1-mb_desp/mb_antes)*100:.0f}% reduccion)')

Memoria ANTES de optimizar:
68.25 MB

  ramo                     :     4 unicos (0.0%) → category
  plan                     :    12 unicos (0.0%) → category
  status_poliza            :     4 unicos (0.0%) → category
  sexo                     :     2 unicos (0.0%) → category
  canal_venta              :     6 unicos (0.0%) → category
  forma_pago               :     4 unicos (0.0%) → category
  estado                   :    15 unicos (0.0%) → category
  estado_civil             :     5 unicos (0.0%) → category
  tipo_vehiculo            :     8 unicos (0.0%) → category

Memoria DESPUES de optimizar:
68.25 MB → 42.18 MB (38% reduccion)


### 📝 Ejercicio 3 — Pipeline de limpieza completo (12 min)

Encapsula todo lo que hicimos en una funcion `limpiar_cartera(ruta_csv)` que:
- Carga con `usecols=ANALITICAS`
- Normaliza `sexo` con str + map
- Convierte fechas con `to_datetime` + `errors='coerce'`
- Calcula `edad_calc`, `dias_vigencia`, `fraccion_expuesta`
- Aplica optimizacion de memoria (categoricas)
- Retorna el DataFrame limpio

Al final llama: `df_limpio = limpiar_cartera('datos/cartera_polizas.csv')`
y verifica que no tenga texto sucio en sexo.

In [21]:
# Tu codigo aqui:
def limpiar_cartera(ruta_csv):

    """Carga archivo CSV de una cartera y realiza limpieza de datos"""
    # Implementa la funcion completa aqui
    # Carga de CSV
    df = pd.read_csv(ruta_csv, usecols=ANALITICAS,na_values=['N/D','N/A','ND','--','Sin dato',''])

    # Normalización de variable categorica
    df['sexo'] = df['sexo'].str.strip().str.upper()
    MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
    }
    df['sexo'] = df['sexo'].map(MAPA_SEXO)

    # Convierte fechas a datetime con manejo de dos formatos distintos
    for col_fecha in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia','fecha_nacimiento']:
        if pd.to_datetime(df[col_fecha], errors='coerce', format='%Y-%m-%d') is not None:
            df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce', format='%Y-%m-%d')
        else:
            df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce', format='%d-%m-%Y')

    # Calcula campos de interés para analisis actuarial
    df['edad_calc'] = ((pd.Timestamp.today() - df['fecha_nacimiento']).dt.days / 365.25)
    df['dias_vigencia'] = (df['fecha_fin_vigencia'] - df['fecha_inicio_vigencia']).dt.days
    df['fraccion_expuesta'] = ( (pd.Timestamp.today() - df['fecha_inicio_vigencia']).dt.days / df['dias_vigencia'] ).round(4)

    # Optimiza memoria, convitiendo a categoricas las columnas con pocos valores unicos
    cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                 'forma_pago','estado','estado_civil','tipo_vehiculo']
    for col in cols_category:
        if col in df.columns:
            n_uniq = df[col].nunique()
            n_tot  = len(df)
            pct = n_uniq/n_tot
            df[col] = df[col].astype('category')
    return df
    pass

base_limpiada = limpiar_cartera('datos/cartera_polizas.csv')
print(base_limpiada.head(3).to_string())
print(base_limpiada["sexo"].value_counts())

    id_poliza     num_poliza    nombre apellido_paterno apellido_materno            rfc fecha_nacimiento  edad sexo estado_civil      ocupacion   ramo         plan fecha_emision fecha_inicio_vigencia fecha_fin_vigencia  num_renovaciones status_poliza motivo_baja    canal_venta marca_vehiculo  modelo_vehiculo tipo_vehiculo  suma_asegurada  deducible  prima_neta  prima_total  forma_pago agente_id           estado  municipio  codigo_postal  edad_calc  dias_vigencia  fraccion_expuesta
0  POL-000001  Vid-21-000001  Gabriela           Moreno             Vega  MOGV020429CG6              NaT    24    F  Union libre  Independiente   Vida     10 anios    2021-11-23            2021-11-23         2022-11-23                 4       Vigente         NaN  Banca Seguros            NaN              NaN           NaN         3000000        NaN     54000.0     67651.20     Mensual     AG054         Veracruz  Poza Rica        36619.0        NaN            365             4.4822
1  POL-000002  Aut-19-000002

---
### Duda 7: ¿Cuándo Cambiar a Polars?

In [22]:
import sys
print(sys.executable)
!pip install polars

c:\Users\derec\anaconda3\envs\diplomado\python.exe


In [23]:
# ── Instalar polars si no esta disponible ────────────────────────────────────
# En tu terminal: pip install polars
# Verificar:
try:
    import polars as pl
    print(f'Polars {pl.__version__} disponible')
    POLARS_OK = True
except ImportError:
    print('Polars no instalado. Ejecuta: pip install polars')
    POLARS_OK = False

Polars 1.40.1 disponible


In [24]:
# ── Comparativa de velocidad pandas vs polars ────────────────────────────────
if POLARS_OK:
    import polars as pl
    import time

    # ── Pandas ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pd = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS)
    res_pd = (df_pd.groupby('ramo')
                   .agg(polizas=('id_poliza','count'),
                        prima_total=('prima_total','sum'),
                        prima_prom=('prima_neta','mean'))
                   .round(2).reset_index())
    t_pd = time.time()-t0

    # ── Polars ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pl = pl.read_csv('datos/cartera_polizas.csv', columns=ANALITICAS)
    res_pl = (df_pl
        .group_by('ramo')
        .agg([
            pl.col('id_poliza').count().alias('polizas'),
            pl.col('prima_total').sum().alias('prima_total'),
            pl.col('prima_neta').mean().alias('prima_prom'),
        ])
    )
    t_pl = time.time()-t0

    print(f'Pandas:  {t_pd*1000:.0f} ms')
    print(f'Polars:  {t_pl*1000:.0f} ms')
    print(f'Polars es {t_pd/t_pl:.1f}x mas rapido en esta operacion')
    print()
    print('Resultado Polars:')
    print(res_pl)
else:
    print('Instala polars para ver la comparativa: pip install polars')

Pandas:  270 ms
Polars:  70 ms
Polars es 3.9x mas rapido en esta operacion

Resultado Polars:
shape: (4, 4)
┌───────────────────────┬─────────┬─────────────┬──────────────┐
│ ramo                  ┆ polizas ┆ prima_total ┆ prima_prom   │
│ ---                   ┆ ---     ┆ ---         ┆ ---          │
│ str                   ┆ u32     ┆ f64         ┆ f64          │
╞═══════════════════════╪═════════╪═════════════╪══════════════╡
│ Accidentes Personales ┆ 5207    ┆ 2.4981e7    ┆ 4002.032838  │
│ Vida                  ┆ 7510    ┆ 4.5433e8    ┆ 50439.777113 │
│ Autos                 ┆ 14752   ┆ 2.5382e8    ┆ 14349.345658 │
│ GMM                   ┆ 22531   ┆ 7.3102e8    ┆ 27065.47908  │
└───────────────────────┴─────────┴─────────────┴──────────────┘


In [ ]:
# ── Sintaxis Polars: las operaciones mas comunes ──────────────────────────────
if POLARS_OK:
    df_pl = pl.read_csv('datos/cartera_polizas.csv',
                        columns=['id_poliza','ramo','prima_total','siniest','zona','edad'])

    # Filtrar
    print('Polizas con prima > 10,000:')
    print(df_pl.filter(pl.col('prima_total') > 10_000).shape)

    # Agregar columna
    df_pl2 = df_pl.with_columns(
        (pl.col('prima_total')/12).alias('prima_mensual'),
        (pl.col('siniest') >= 2).alias('alto_riesgo'),
    )

    # Sort
    print('Top 5 primas:')
    print(df_pl2.sort('prima_total', descending=True).head(5)[['id_poliza','ramo','prima_total']])

    # Lazy evaluation — ejecuta todo optimizado al final
    resultado = (
        pl.scan_csv('datos/cartera_polizas.csv')  # NO carga en memoria aun
        .filter(pl.col('prima_total') > 5000)
        .group_by('ramo')
        .agg(pl.col('prima_total').sum())
        .collect()  # AHORA ejecuta con el plan optimizado
    )
    print('Lazy evaluation result:')
    print(resultado)

Polizas con prima > 10,000:
(33919, 4)


ColumnNotFoundError: unable to find column "siniest"; valid columns: ["id_poliza", "edad", "ramo", "prima_total"]

### Regla practica: ¿Pandas o Polars?

| Situacion | Usa |
|-----------|-----|
| Aprendizaje, primeros modelos | **pandas** — el ecosistema es enorme |
| < 1 millon de filas | **pandas** — mas que suficiente |
| Analisis interactivo en notebook | **pandas** — syntax mas conocida |
| Pipeline de produccion > 5M filas | **polars** — 5-20x mas rapido |
| Ingesta diaria de datos grandes | **polars** con lazy evaluation |
| ETL de empresa | **polars** o **spark** segun el tamano |

---
## pivot_table con Datos Reales


In [28]:
# ── Agregar columnas necesarias para el pivot ────────────────────────────────
df_work = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS,
                      na_values=['N/D','N/A',''])
df_work['prima_neta'] = df_work['prima_neta'].fillna(df_work.groupby('ramo')['prima_neta'].transform('median'))
df_work['g_edad'] = pd.cut(df_work['edad'], bins=[0,30,45,60,100],
                           labels=['18-30','31-45','46-60','61+'])
df_work['siniest_flag'] = (df_work['prima_neta'] > df_work['prima_neta'].quantile(0.75)).astype(int)

print(f'Dataset para pivot: {df_work.shape}')

Dataset para pivot: (50000, 34)


In [29]:
# ── pivot_table: prima por ramo x grupo de edad ──────────────────────────────
tabla_prima = pd.pivot_table(
    df_work,
    values   = 'prima_total',
    index    = 'ramo',
    columns  = 'g_edad',
    aggfunc  = 'sum',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
).round(0) / 1_000  # en miles de pesos

print('Prima total por ramo y grupo de edad (miles MXN):')
print(tabla_prima.to_string())

Prima total por ramo y grupo de edad (miles MXN):
g_edad                      18-30       31-45       46-60         61+        TOTAL
ramo                                                                              
Accidentes Personales    5248.369    8348.395    8731.668    2652.247    24980.679
Autos                   55457.022   84225.512   84571.404   29569.569   253823.506
GMM                    139278.258  223908.688  264218.298  103617.078   731022.322
Vida                    79667.849  134774.641  166301.773   73587.860   454332.123
TOTAL                  279651.498  451257.236  523823.143  209426.754  1464158.631


C:\Users\derec\AppData\Local\Temp\ipykernel_8140\3938379720.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  tabla_prima = pd.pivot_table(


In [30]:
# ── pivot_table: polizas por zona x canal ────────────────────────────────────
tabla_canal = pd.pivot_table(
    df_work,
    values   = 'id_poliza',
    index    = 'estado',
    columns  = 'canal_venta',
    aggfunc  = 'count',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
)

print('Polizas por estado y canal de venta:')
print(tabla_canal.to_string())

Polizas por estado y canal de venta:
canal_venta       Agente  Banca Seguros  Broker  Digital  Directo  Promotor  TOTAL
estado                                                                            
Baja California     1646            329     678      241      356        98   3348
CDMX                1719            311     684      217      318       108   3357
Chihuahua           1656            344     642      250      357       101   3350
Coahuila            1721            349     672      199      329        96   3366
Estado de Mexico    1703            336     652      253      287        93   3324
Guanajuato          1745            339     669      237      359        96   3445
Jalisco             1664            322     670      258      332       106   3352
Michoacan           1621            327     711      217      319        99   3294
Nuevo Leon          1681            335     615      225      326        96   3278
Puebla              1586            330     672   

---
## Exportar — CSV, Excel, Parquet y JSON


In [31]:
# ── Guardar en todos los formatos y comparar ──────────────────────────────────
df_export = df_work.head(10_000)  # subconjunto para demo

formatos = {
    'CSV':     ('datos/export_demo.csv',
                lambda: df_export.to_csv('datos/export_demo.csv', index=False)),
    'Parquet': ('datos/export_demo.parquet',
                lambda: df_export.to_parquet('datos/export_demo.parquet', index=False)),
    'JSON':    ('datos/export_demo.json',
                lambda: df_export.to_json('datos/export_demo.json',
                                          orient='records', force_ascii=False)),
}

print(f'{"Formato":<10} {"Tamano":>10} {"Tiempo":>10}')
print('-' * 35)
for nombre, (ruta, guardar) in formatos.items():
    t0 = time.time()
    guardar()
    t = (time.time()-t0)*1000
    kb = os.path.getsize(ruta)/1024
    print(f'{nombre:<10} {kb:>8.0f} KB {t:>8.0f} ms')

Formato        Tamano     Tiempo
-----------------------------------
CSV            2465 KB      100 ms
Parquet         633 KB      121 ms
JSON           7645 KB       42 ms


In [32]:
# ── Excel multihoja — el entregable mas solicitado en empresas ───────────────
resumen_ramo = df_work.groupby('ramo').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum'),
    prima_prom =('prima_total','mean'),
).round(2).reset_index()
resumen_ramo['pct_cartera'] = (resumen_ramo['prima_total']/resumen_ramo['prima_total'].sum()*100).round(1)

with pd.ExcelWriter('datos/reporte_demo.xlsx', engine='openpyxl') as writer:
    df_work.head(5_000).to_excel(writer, sheet_name='Cartera', index=False)
    resumen_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    tabla_prima.to_excel(writer, sheet_name='Pivot_Prima')
    tabla_canal.to_excel(writer, sheet_name='Pivot_Canal')

kb_xl = os.path.getsize('datos/reporte_demo.xlsx')/1024
print(f'Excel multihoja generado: {kb_xl:.0f} KB con 4 hojas')

Excel multihoja generado: 972 KB con 4 hojas


---
## 🏆 Ejercicio Integrador Final — Pipeline Completo

**Contexto:** Tu jefa de estadistica te pide el reporte ejecutivo del Q1 2026.
Tienes los 4 archivos crudos. Debes construir un pipeline completo,
documentando cada decision de limpieza.

**Tiempo:** 40 minutos  |  **Entregable:** Excel con 5 hojas + Parquet

---

### Criterios de evaluacion:
- ✅ Cada decision de descarte de columnas esta justificada con comentario
- ✅ No hay texto sucio en columnas categoricas clave
- ✅ Todas las fechas son datetime, no object
- ✅ dtypes optimizados sin perder precision en montos
- ✅ El Excel tiene las 5 hojas con los datos correctos
- ✅ El Parquet es mas pequeno que el CSV equivalente

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 1: INGESTA INTELIGENTE
# ══════════════════════════════════════════════════════════════════════════════
# Carga cartera, siniestros y catalogo de ramos/agentes.
# Usa SOLO las columnas que necesitas — justifica con comentario.
# Mide la memoria ahorrada vs cargar todo.

# Tu codigo aqui:


In [82]:
# id_ Poliza: es el identificador unico de cada poliza, util para revisar toda la información relacionada, como el seguro junto con caracteristica y los siniestros
# fecha_nacimiento: para ramos de vida, Accidentes personales y GMM.
# sexo: segregación del riesgo, parra ramos de vida y GMM.
# Ocupacion y nivel_educativo: para ramos de vida, Accidentes personales y GMM.
# ramo: para identificar el tipo de seguro.
# plan: información del producto, podemos hacer analisis de costos, siniestralidad, bajas, etc.
# fecha_emision, fecha_inicio_vigencia y fecha_fin_vigencia: para calculos de exposición, devengamientos, etc.
# num_renovaciones: para analizar la retención de clientes.
# Estatus_poliza: para analizar la cartera activa vs cancelada.
# motivo_baja: para analizar las razones de cancelación.
# canal_venta, agente_id: para analizar el desempeño de cada canal y los agentes.
# marca_vehiculo, modelo_vehiculo y tipo_vehiculo: para analisis de ramos de autos.
# suma_asegurada, deducible, prima_neta y prima_total: para analisis financieros y de siniestralidad.
# cuota_prima y num_cuotas: para analisis de formas de pago y su impacto en la cartera.
# Estado, municipio y codigo_postal: para analisis geoespacial del riesgo.
# agente_id: Para vincularlo con la información de los agentes.

df_full = pd.read_csv('datos/cartera_polizas.csv')
mb_full = df_full.memory_usage(deep=True).sum() / 1024**2

ANALISIS = ['id_poliza', 'fecha_nacimiento','nombre', 'sexo', 'ocupacion', 'nivel_educacion', 'ramo', 'plan',
'fecha_emision', 'fecha_inicio_vigencia', 'fecha_fin_vigencia', 'num_renovaciones', 'status_poliza', 
'motivo_baja', 'marca_vehiculo', 'modelo_vehiculo', 'tipo_vehiculo', 'suma_asegurada', 'deducible', 
'prima_neta', 'prima_total', 'cuota_prima', 'forma_pago', 'estado', 'municipio', 'codigo_postal','agente_id','edad']

t0 = time.time()
cartera = pd.read_csv( 'datos/cartera_polizas.csv' ,usecols=ANALISIS, na_values=['N/D','N/A','ND','--','Sin dato',''])
t1 = time.time()
mb_opt = cartera.memory_usage(deep=True).sum() / 1024**2

print('=== COMPARATIVA DE CARGA ===')
print(f'Carga completa (46 cols):  {mb_full:.1f} MB')
print(f'Carga selectiva ({len(ANALISIS)} cols):  {mb_opt:.1f} MB')
print(f'Reduccion de memoria:      {(1-mb_opt/mb_full)*100:.0f}%')
print(f'Tiempo de carga:           {(t1-t0)*1000:.0f} ms')
print()
print(f'Dataset de trabajo: {cartera.shape}')
cartera


=== COMPARATIVA DE CARGA ===
Carga completa (46 cols):  112.0 MB
Carga selectiva (28 cols):  58.4 MB
Reduccion de memoria:      48%
Tiempo de carga:           199 ms

Dataset de trabajo: (50000, 28)


,id_poliza,nombre,fecha_nacimiento,edad,sexo,ocupacion,nivel_educacion,ramo,plan,fecha_emision,...,suma_asegurada,deducible,prima_neta,prima_total,cuota_prima,forma_pago,agente_id,estado,municipio,codigo_postal
0,POL-000001,Gabriela,29/04/2002,24,F,Independiente,Licenciatura,Vida,10 anios,2021-11-23,...,3000000,NaN,54000.0,67651.20,5637.60,Mensual,AG054,Veracruz,Poza Rica,36619.0
1,POL-000002,Valeria,15/08/2002,23,Femenino,Contador,Licenciatura,Autos,Amplia Plus,2019-08-19,...,150000,8000.0,5250.0,6394.50,1598.62,Trimestral,AG004,Michoacan,Morelia,58889.0
2,POL-000003,Fernanda,18/10/1994,31,M,Ingeniero,Licenciatura,GMM,Basico,2022-07-11,...,800000,5000.0,17600.0,22049.28,1837.44,Mensual,AG051,Baja California,Tecate,45784.0
3,POL-000004,Silvia,04/02/1986,40,Masculino,Arquitecto,Licenciatura,Vida,20 anios,2019-03-12,...,5000000,NaN,99000.0,124027.20,10335.60,Mensual,AG026,CDMX,Coyoacan,53982.0
4,POL-000005,Antonio,22/07/1996,29,F,Independiente,Posgrado,Vida,20 anios,2020-11-03,...,3000000,NaN,54000.0,62640.00,62640.00,Anual,AG050,Guanajuato,Guanajuato,69777.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,POL-049996,Pedro,01/11/2000,25,F,Ama de casa,Licenciatura,Accidentes Personales,Individual,2020-01-17,...,300000,3000.0,3600.0,4510.08,375.84,Mensual,AG004,Puebla,Izucar,58300.0
49996,POL-049997,Enrique,23/10/1972,53,M,Medico,Preparatoria,GMM,Plus,2024-09-03,...,500000,8000.0,13145.0,15248.20,15248.20,Anual,AG072,Yucatan,Hunucma,50556.0
49997,POL-049998,Irene,15/01/1988,38,M,Medico,Preparatoria,Autos,Amplia,2021-02-10,...,200000,8000.0,7000.0,8363.60,4181.80,Semestral,AG028,Puebla,Cholula,93023.0
49998,POL-049999,Rosa,28/05/1995,30,F,Empresario,Licenciatura,Accidentes Personales,Individual,2024-03-05,...,500000,1000.0,6000.0,6960.00,6960.00,Anual,AG064,Sonora,Nogales,14656.0


In [83]:
# id_siniestro: Es el identificador único de cada siniestro
# id_poliza: Para vincular los siniestros con cada poliza
# fecha_ocurrencia, fecha_reporte, fecha_apertura, fecha_ultimo_movimiento y fecha_cierre: Identificar el comportamiento del siniestro a lo largo del tiempo
# dias_resolución, días_reporte
# tipo_siniestro subtipo, descripción_diagnostico, hospital_id, status_siniestro: Obtener información relevante de los siniestros
# monto_reclamado, monto_deducible_aplic, monto_coaseguro, monto_pagado, monto_reservado y moneda: Para analisis de siniestralidad
# motivo_rechazo.
ANALISIS_SIN = ['id_siniestro','id_poliza','fecha_ocurrencia', 'fecha_reporte', 'fecha_apertura', 'fecha_ultimo_movimiento', 'fecha_cierre','dias_resolucion', 'dias_reporte',
            'tipo_siniestro', 'subtipo', 'descripcion_diagnostico', 'hospital_id', 'status_siniestro','monto_reclamado', 'monto_deducible_aplic', 'monto_coaseguro', 'monto_pagado',
              'monto_reservado', 'moneda', 'motivo_rechazo']

df_full = pd.read_csv('datos/siniestros.csv')
mb_full = df_full.memory_usage(deep=True).sum() / 1024**2

t0 = time.time()
siniestros = pd.read_csv('datos/siniestros.csv', usecols = ANALISIS_SIN ,na_values=['N/D','N/A','ND','--','Sin dato',''])

t1 = time.time()
mb_opt = siniestros.memory_usage(deep=True).sum() / 1024**2

print('=== COMPARATIVA DE CARGA ===')
print(f'Carga completa (46 cols):  {mb_full:.1f} MB')
print(f'Carga selectiva ({len(ANALISIS_SIN)} cols):  {mb_opt:.1f} MB')
print(f'Reduccion de memoria:      {(1-mb_opt/mb_full)*100:.0f}%')
print(f'Tiempo de carga:           {(t1-t0)*1000:.0f} ms')
print()
print(f'Dataset de trabajo: {siniestros.shape}')

siniestros


=== COMPARATIVA DE CARGA ===
Carga completa (46 cols):  26.7 MB
Carga selectiva (21 cols):  16.0 MB
Reduccion de memoria:      40%
Tiempo de carga:           63 ms

Dataset de trabajo: (18000, 21)


,id_siniestro,id_poliza,fecha_ocurrencia,fecha_reporte,fecha_apertura,fecha_ultimo_movimiento,fecha_cierre,dias_reporte,dias_resolucion,tipo_siniestro,...,descripcion_diagnostico,hospital_id,monto_reclamado,monto_deducible_aplic,monto_coaseguro,monto_pagado,monto_reservado,moneda,status_siniestro,motivo_rechazo
0,SIN-0000001,POL-007682,2022-06-22,2022-06-25,25/06/2022,11/08/2022,NaN,3,NaN,Robo total,...,Diabetes complicada,Taller,23693.41,3000.0,2069.34,0.00,18954.73,MXN,En proceso,NaN
1,SIN-0000002,POL-049056,2023-04-07,2023-04-14,14/04/2023,11/06/2023,NaN,7,NaN,Urgencia,...,Parto,HSP027,14545.50,0.0,1454.55,0.00,11636.40,MXN,En proceso,NaN
2,SIN-0000003,POL-035191,2022-05-24,2022-05-26,26/05/2022,16/06/2022,NaN,2,NaN,Invalidez parcial,...,Infarto,NaN,94047.09,5000.0,8904.71,0.00,75237.67,MXN,En proceso,NaN
3,SIN-0000004,POL-035037,2021-10-29,2021-10-30,30/10/2021,10/11/2021,NaN,1,NaN,Cristales,...,Neumonia,Taller,10992.47,0.0,1099.25,0.00,8793.98,MXN,En proceso,NaN
4,SIN-0000005,POL-041900,2023-11-15,2023-11-18,18/11/2023,19/12/2023,2024-05-16,3,183.0,Accidente,...,Cataratas,NaN,1314.78,15000.0,0.00,0.00,0.00,USD,Pagado,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17995,SIN-0017996,POL-032046,2020-12-25,2021-01-08,08/01/2021,04/03/2021,NaN,14,NaN,Fallecimiento,...,Infarto,NaN,13779.39,8000.0,577.94,0.00,11023.51,MXN,En proceso,NaN
17996,SIN-0017997,POL-049150,2022-12-10,2022-12-15,15/12/2022,05/02/2023,2023-01-14,5,35.0,Responsabilidad civil,...,Apendicitis,Taller,22807.60,3000.0,1980.76,17826.84,0.00,MXN,Pagado,NaN
17997,SIN-0017998,POL-044066,2023-09-21,2023-10-21,21/10/2023,28/11/2023,NaN,30,NaN,Invalidez total,...,Accidente,NaN,26200.07,8000.0,1820.01,0.00,20960.06,MXN,En proceso,NaN
17998,SIN-0017999,POL-026377,2020-09-04,2020-09-07,07/09/2020,26/09/2020,NaN,3,NaN,Hospitalizacion,...,Hernia,HSP025,5384.39,3000.0,238.44,0.00,4307.51,MXN,En proceso,NaN


In [84]:
# Todas las columnas son relevantes, obtenemos información general de cada ramo, a lo mejor para no repetir, se omitiria la columna de ramo, ya que, ya se encuentra en las demás bases.

catalogo_ramos = pd.read_csv('datos/catalogo_ramos.csv', na_values=['N/D','N/A','ND','--','Sin dato',''])
catalogo_ramos


,ramo_id,ramo,nombre_largo,tasa_base,deducible_base,coaseguro_pct,tope_coaseguro,requiere_medico,reasegurado,activo
0,R01,GMM,Gastos Medicos Mayores,0.022,5000,0.1,50000,True,True,True
1,R02,Autos,Automoviles,0.035,3000,0.0,0,False,False,True
2,R03,Vida,Vida Individual,0.018,0,0.0,0,True,True,True
3,R04,Accidentes Personales,Accidentes Personales,0.012,1000,0.1,20000,False,False,True


In [85]:
# Todas la columnas de la tabla son relevantes para hacer analisis respecto a los agentes vinculados a la aseguradora.
catalogo_agentes = pd.read_csv('datos/catalogo_agentes.csv', na_values=['N/D','N/A','ND','--','Sin dato',''])
catalogo_agentes

,agente_id,nombre,tipo,region,clave_amis,fecha_alta,activo,comision_pct,cartera_polizas,meta_anual
0,AG001,Jorge Chavez,Independiente,Jalisco,AMIS-86105,2010-01-01,True,0.0679,127,1000000
1,AG002,Norma Navarro,Independiente,Guanajuato,AMIS-81642,2010-03-11,True,0.0792,538,3000000
2,AG003,Daniel Sanchez,Promotor,Queretaro,AMIS-20887,2010-05-19,True,0.0866,639,500000
3,AG004,Patricia Rios,Independiente,Veracruz,AMIS-89971,2010-07-28,True,0.1380,85,3000000
4,AG005,Claudia Salazar,Independiente,Estado de Mexico,AMIS-36056,2010-10-05,False,0.0772,393,3000000
...,...,...,...,...,...,...,...,...,...,...
75,AG076,Elena Diaz,Empresa,Estado de Mexico,AMIS-88409,2024-03-28,True,0.0771,653,1000000
76,AG077,Oscar Moreno,Independiente,Chihuahua,AMIS-40582,2024-06-05,True,0.0717,279,3000000
77,AG078,Luis Aguilar,Independiente,Veracruz,AMIS-98932,2024-08-14,True,0.1270,589,3000000
78,AG079,Sofia Perez,Promotor,Sonora,AMIS-19110,2024-10-22,False,0.1055,435,500000


In [86]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 2: LIMPIEZA COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
# 2a. Elimina duplicados de cartera
# 2b. Normaliza sexo con str.strip().str.upper() + .map(MAPA_SEXO)
# 2c. Convierte TODAS las fechas a datetime con errors='coerce'
# 2d. Rellena NaN de prima_neta con la MEDIANA POR RAMO (no global)
#     df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# 2e. Limpia codigo_postal (reemplaza 'N/D' con NaN)
# 2f. Aplica optimizacion de categoricas (sin tocar float64 de primas)

# Tu codigo aqui:
# Definimos una función que te haga la limpieza

def limpieza_datos(df):
    """ Función que permite limpiar la base de Cartera o siniestros"""
    # 2a. Elimina duplicados de df
    print("=== ELIMINANDO DUPLICADOS ===")
    df = df.drop_duplicates()
    print("=== NORMALIZAR LA COLUMNA SEXO ===")
    try:
        MAPA_SEXO = {'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
                    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',}
        df['sexo'] = df['sexo'].str.strip().str.upper().map(MAPA_SEXO)
    except Exception as e: 
        print(f"No se identifica la columna {e}")
    print("=== CONVERTIR COLUMNAS FECHA A DATETIME ===")

    col_fechas = [
        # Columnas de cartera
        "fecha_emision", "fecha_inicio_vigencia", "fecha_fin_vigencia", "fecha_nacimiento",
        # Columnas de siniestro
        "fecha_ocurrencia", "fecha_reporte", "fecha_apertura", "fecha_ultimo_movimiento", "fecha_cierre"
    ]

    for col in col_fechas:
        # Verificamos si la columna realmente existe en el DataFrame antes de operar
        if col in df.columns:
            # 'mixed' detecta automáticamente si viene como YYYY-MM-DD o DD/MM/YYYY en cada celda
            df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed')
        else:
            print(f"No existe la columna: {col}")

    if 'prima_neta' in df.columns:
        print("=== RELLENAR NaN DE PRIMA_NETA con la MEDIANA POR RAMO ===")
        df['prima_neta'] =  df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
    if 'codigo_postal' in df.columns:
        print("=== LIMPIAR CODIGO POSTAL ===")
        df['codigo_postal'] = df['codigo_postal'].replace('N/D',np.nan)

    umbral_cat=0.05
        # Categoricas
    for col in df.select_dtypes('object').columns:
        pct = df[col].nunique() / len(df)
        if pct < umbral_cat:
            df[col] = df[col].astype('category')
    return df

    


In [87]:
df_limpio = limpieza_datos(cartera)
df_limpio

=== ELIMINANDO DUPLICADOS ===
=== NORMALIZAR LA COLUMNA SEXO ===
=== CONVERTIR COLUMNAS FECHA A DATETIME ===
No existe la columna: fecha_ocurrencia
No existe la columna: fecha_reporte
No existe la columna: fecha_apertura
No existe la columna: fecha_ultimo_movimiento
No existe la columna: fecha_cierre
=== RELLENAR NaN DE PRIMA_NETA con la MEDIANA POR RAMO ===
=== LIMPIAR CODIGO POSTAL ===


,id_poliza,nombre,fecha_nacimiento,edad,sexo,ocupacion,nivel_educacion,ramo,plan,fecha_emision,...,suma_asegurada,deducible,prima_neta,prima_total,cuota_prima,forma_pago,agente_id,estado,municipio,codigo_postal
0,POL-000001,Gabriela,2002-04-29,24,F,Independiente,Licenciatura,Vida,10 anios,2021-11-23,...,3000000,NaN,54000.0,67651.20,5637.60,Mensual,AG054,Veracruz,Poza Rica,36619.0
1,POL-000002,Valeria,2002-08-15,23,F,Contador,Licenciatura,Autos,Amplia Plus,2019-08-19,...,150000,8000.0,5250.0,6394.50,1598.62,Trimestral,AG004,Michoacan,Morelia,58889.0
2,POL-000003,Fernanda,1994-10-18,31,M,Ingeniero,Licenciatura,GMM,Basico,2022-07-11,...,800000,5000.0,17600.0,22049.28,1837.44,Mensual,AG051,Baja California,Tecate,45784.0
3,POL-000004,Silvia,1986-04-02,40,M,Arquitecto,Licenciatura,Vida,20 anios,2019-03-12,...,5000000,NaN,99000.0,124027.20,10335.60,Mensual,AG026,CDMX,Coyoacan,53982.0
4,POL-000005,Antonio,1996-07-22,29,F,Independiente,Posgrado,Vida,20 anios,2020-11-03,...,3000000,NaN,54000.0,62640.00,62640.00,Anual,AG050,Guanajuato,Guanajuato,69777.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,POL-049996,Pedro,2000-01-11,25,F,Ama de casa,Licenciatura,Accidentes Personales,Individual,2020-01-17,...,300000,3000.0,3600.0,4510.08,375.84,Mensual,AG004,Puebla,Izucar,58300.0
49996,POL-049997,Enrique,1972-10-23,53,M,Medico,Preparatoria,GMM,Plus,2024-09-03,...,500000,8000.0,13145.0,15248.20,15248.20,Anual,AG072,Yucatan,Hunucma,50556.0
49997,POL-049998,Irene,1988-01-15,38,M,Medico,Preparatoria,Autos,Amplia,2021-02-10,...,200000,8000.0,7000.0,8363.60,4181.80,Semestral,AG028,Puebla,Cholula,93023.0
49998,POL-049999,Rosa,1995-05-28,30,F,Empresario,Licenciatura,Accidentes Personales,Individual,2024-03-05,...,500000,1000.0,6000.0,6960.00,6960.00,Anual,AG064,Sonora,Nogales,14656.0


In [88]:
siniestros_limpio = limpieza_datos(siniestros)
siniestros_limpio

=== ELIMINANDO DUPLICADOS ===
=== NORMALIZAR LA COLUMNA SEXO ===
No se identifica la columna 'sexo'
=== CONVERTIR COLUMNAS FECHA A DATETIME ===
No existe la columna: fecha_emision
No existe la columna: fecha_inicio_vigencia
No existe la columna: fecha_fin_vigencia
No existe la columna: fecha_nacimiento


,id_siniestro,id_poliza,fecha_ocurrencia,fecha_reporte,fecha_apertura,fecha_ultimo_movimiento,fecha_cierre,dias_reporte,dias_resolucion,tipo_siniestro,...,descripcion_diagnostico,hospital_id,monto_reclamado,monto_deducible_aplic,monto_coaseguro,monto_pagado,monto_reservado,moneda,status_siniestro,motivo_rechazo
0,SIN-0000001,POL-007682,2022-06-22,2022-06-25,2022-06-25,2022-11-08,NaT,3,NaN,Robo total,...,Diabetes complicada,Taller,23693.41,3000.0,2069.34,0.00,18954.73,MXN,En proceso,NaN
1,SIN-0000002,POL-049056,2023-04-07,2023-04-14,2023-04-14,2023-11-06,NaT,7,NaN,Urgencia,...,Parto,HSP027,14545.50,0.0,1454.55,0.00,11636.40,MXN,En proceso,NaN
2,SIN-0000003,POL-035191,2022-05-24,2022-05-26,2022-05-26,2022-06-16,NaT,2,NaN,Invalidez parcial,...,Infarto,NaN,94047.09,5000.0,8904.71,0.00,75237.67,MXN,En proceso,NaN
3,SIN-0000004,POL-035037,2021-10-29,2021-10-30,2021-10-30,2021-10-11,NaT,1,NaN,Cristales,...,Neumonia,Taller,10992.47,0.0,1099.25,0.00,8793.98,MXN,En proceso,NaN
4,SIN-0000005,POL-041900,2023-11-15,2023-11-18,2023-11-18,2023-12-19,2024-05-16,3,183.0,Accidente,...,Cataratas,NaN,1314.78,15000.0,0.00,0.00,0.00,USD,Pagado,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17995,SIN-0017996,POL-032046,2020-12-25,2021-01-08,2021-08-01,2021-04-03,NaT,14,NaN,Fallecimiento,...,Infarto,NaN,13779.39,8000.0,577.94,0.00,11023.51,MXN,En proceso,NaN
17996,SIN-0017997,POL-049150,2022-12-10,2022-12-15,2022-12-15,2023-05-02,2023-01-14,5,35.0,Responsabilidad civil,...,Apendicitis,Taller,22807.60,3000.0,1980.76,17826.84,0.00,MXN,Pagado,NaN
17997,SIN-0017998,POL-044066,2023-09-21,2023-10-21,2023-10-21,2023-11-28,NaT,30,NaN,Invalidez total,...,Accidente,NaN,26200.07,8000.0,1820.01,0.00,20960.06,MXN,En proceso,NaN
17998,SIN-0017999,POL-026377,2020-09-04,2020-09-07,2020-07-09,2020-09-26,NaT,3,NaN,Hospitalizacion,...,Hernia,HSP025,5384.39,3000.0,238.44,0.00,4307.51,MXN,En proceso,NaN


In [89]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 3: ENRIQUECIMIENTO
# ══════════════════════════════════════════════════════════════════════════════
# 3a. Merge con catalogo_ramos: agregar nombre_largo, tasa_base
# 3b. Merge con catalogo_agentes: agregar nombre del agente, region
# 3c. Crear: g_edad con pd.cut
# 3d. Crear: prima_calc = suma_asegurada * tasa_base * 1.16
# 3e. Crear: nivel_riesgo con .apply(clasificar_riesgo) — de mi_modulo
# 3f. Crear: edad_calc desde fecha_nacimiento
# 3g. Crear: dias_vigencia, fraccion_expuesta

from mi_modulo import clasificar_riesgo

# Tu codigo aqui:

# 3a. Merge con catalogo_ramos: agregar nombre_largo, tasa_base
cartera_ramo = pd.merge(df_limpio, catalogo_ramos[['ramo','nombre_largo','tasa_base']], on = 'ramo', how = 'left')

# 3b. Merge con catalogo_agentes: agregar nombre del agente, region
df_final = pd.merge(cartera_ramo, catalogo_agentes[['agente_id','nombre','region']], on = 'agente_id', how = 'left', suffixes=('', '_agente'))

# 3c. Crear: g_edad con pd.cut
df_final['g_edad'] = pd.cut(
    df_final['edad'],
    bins=[0,30,45,60,100],
    labels=['18-30', '31-45', '46-60', '61+']
)

# 3d. Crear: prima_calc = suma_asegurada * tasa_base * 1.16
df_final['prima_calc'] = df_final['suma_asegurada'] * df_final['tasa_base'] * 1.16

# 3e. Crear: nivel_riesgo con .apply(clasificar_riesgo) — de mi_modulo
# Para obtener la cantididad de siniestros, necesitamos hacer el resumen por id_poliza
siniest = siniestros.groupby('id_poliza').agg(
    Siniestros = ('id_siniestro', 'count'),
    monto_reclamado_total = ('monto_reclamado', 'sum')
)
# Hacemos un merge para obtener la cantidad de siniestros
df_final = pd.merge(df_final, siniest, on='id_poliza', how= 'left')
# Aplicamos la función
df_final['nivel_riesgo'] = df_final['Siniestros'].apply(clasificar_riesgo)

# 3f. Crear: edad_calc desde fecha_nacimiento
df_final['edad_calc'] = ((pd.Timestamp.today()- df_final['fecha_nacimiento']).dt.days/365).round(1)

# 3g. Crear: dias_vigencia, fraccion_expuesta
df_final['dias_vigencia'] = (df_final['fecha_fin_vigencia'] - df_final['fecha_inicio_vigencia']).dt.days

# Clip limita el valor de los registros entre 0 y 1
df_final['fraccion_expuesta'] = (((pd.Timestamp.today()-df_final['fecha_inicio_vigencia']).dt.days)/df_final['dias_vigencia']).clip(0,1)

df_final

,id_poliza,nombre,fecha_nacimiento,edad,sexo,ocupacion,nivel_educacion,ramo,plan,fecha_emision,...,nombre_agente,region,g_edad,prima_calc,Siniestros,monto_reclamado_total,nivel_riesgo,edad_calc,dias_vigencia,fraccion_expuesta
0,POL-000001,Gabriela,2002-04-29,24,F,Independiente,Licenciatura,Vida,10 anios,2021-11-23,...,Enrique Castillo,Chihuahua,18-30,62640.0,NaN,NaN,BAJO,24.1,365,1.0
1,POL-000002,Valeria,2002-08-15,23,F,Contador,Licenciatura,Autos,Amplia Plus,2019-08-19,...,Patricia Rios,Veracruz,18-30,6090.0,NaN,NaN,BAJO,23.8,366,1.0
2,POL-000003,Fernanda,1994-10-18,31,M,Ingeniero,Licenciatura,GMM,Basico,2022-07-11,...,Manuel Mendoza,Chihuahua,31-45,20416.0,NaN,NaN,BAJO,31.6,365,1.0
3,POL-000004,Silvia,1986-04-02,40,M,Arquitecto,Licenciatura,Vida,20 anios,2019-03-12,...,Esperanza Jimenez,Nuevo Leon,31-45,104400.0,NaN,NaN,BAJO,40.2,366,1.0
4,POL-000005,Antonio,1996-07-22,29,F,Independiente,Posgrado,Vida,20 anios,2020-11-03,...,Gabriela Rios,Nuevo Leon,18-30,62640.0,1.0,38260.15,MEDIO,29.8,365,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,POL-049996,Pedro,2000-01-11,25,F,Ama de casa,Licenciatura,Accidentes Personales,Individual,2020-01-17,...,Patricia Rios,Veracruz,18-30,4176.0,NaN,NaN,BAJO,26.4,366,1.0
49996,POL-049997,Enrique,1972-10-23,53,M,Medico,Preparatoria,GMM,Plus,2024-09-03,...,Carlos Romero,Puebla,46-60,12760.0,2.0,57919.14,MEDIO,53.6,365,1.0
49997,POL-049998,Irene,1988-01-15,38,M,Medico,Preparatoria,Autos,Amplia,2021-02-10,...,Hector Perez,Coahuila,31-45,8120.0,2.0,43774.30,MEDIO,38.4,365,1.0
49998,POL-049999,Rosa,1995-05-28,30,F,Empresario,Licenciatura,Accidentes Personales,Individual,2024-03-05,...,Fernando Medina,Baja California,18-30,6960.0,NaN,NaN,BAJO,31.0,365,1.0


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 4: ANALISIS Y REPORTES
# ══════════════════════════════════════════════════════════════════════════════
# 4a. groupby+agg por ramo: polizas, prima_total, prima_prom, pct_cartera
# 4b. groupby+agg por agente: polizas, prima_total, comision (10%)
# 4c. pivot_table prima por ramo x g_edad con margins=True
# 4d. pivot_table polizas por estado x ramo
# 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia

# Tu codigo aqui:

# 4a. groupby+agg por ramo: polizas, prima_total, prima_prom, pct_cartera
resumen_ramo = df_final.groupby('ramo').agg(
    polizas = ('id_poliza', 'count'),
    prima_total = ('prima_total', 'sum'),
    prima_pro = ('prima_total', 'mean'),
).round(0).reset_index().sort_values('prima_total', ascending=False)
resumen_ramo['pct_cartera'] = (resumen_ramo['prima_total']/ resumen_ramo['prima_total'].sum()*100).round(2)

print(' '*30 +'Resumen por ramo')
print(resumen_ramo.to_string(index=False))

# 4b. groupby+agg por agente: polizas, prima_total, comision (10%)
resumen_agente = df_final.groupby('nombre_agente').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum')
).round(0).reset_index().sort_values('prima_total', ascending=False)
resumen_agente['comisión'] = resumen_agente['prima_total']*0.10

print('='*65)
print(' '*30 +'Resumen por agente:')
print(resumen_agente.to_string(index=False))




                              Resumen por ramo
                 ramo  polizas  prima_total  prima_pro  pct_cartera
                  GMM    22531  731022322.0    32445.0        49.93
                 Vida     7510  454332123.0    60497.0        31.03
                Autos    14752  253823506.0    17206.0        17.34
Accidentes Personales     5207   24980679.0     4798.0         1.71
                              Resumen por agente:
    nombre_agente  polizas  prima_total  comisión
      Sofia Perez     1231   37442770.0 3744277.0
      Jose Medina     1227   34800214.0 3480021.4
   Victor Jimenez     1207   34242602.0 3424260.2
  Pablo Dominguez      674   21039960.0 2103996.0
   Gabriela Nunez      662   20690915.0 2069091.5
     Monica Silva      658   20375763.0 2037576.3
    Sandra Garcia      679   20269988.0 2026998.8
     Raul Navarro      645   20155985.0 2015598.5
     Diana Molina      662   20082984.0 2008298.4
       Elena Diaz      668   20065958.0 2006595.8
   Daniel San

In [ ]:
# 4c. pivot_table prima por ramo x g_edad con margins=True
pivot_prima = pd.pivot_table(
        df_final, values='prima_total', index='ramo',
        columns='g_edad', aggfunc='sum', fill_value=0, 
        margins=True, margins_name='TOTAL'
    ).round(0)/ 1000000 
print(' '*30 +'Prima por ramo y grupo de edad')
print(pivot_prima.to_string())

# 4d. pivot_table polizas por estado x ramo
pivot_estado = pd.pivot_table(
        df_final, values='id_poliza', index='estado',
        columns='ramo', aggfunc='count', fill_value=0, margins=True, margins_name='TOTAL'
    ).reset_index()
print("="*90)
print(' '*30 +'Polizas por estado y ramo')
print(pivot_estado.to_string())


# 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia
print("="*90)
print(f"Ramo con mayor prima total es: {resumen_ramo.loc[resumen_ramo['prima_total'].idxmax(), 'ramo']}")

df_estados = pivot_estado[pivot_estado['estado'] != 'TOTAL']
print(f"Estado con mayor Polizas es: {df_estados.loc[df_estados['TOTAL'].idxmax(), 'estado']}")



                              Prima por ramo y grupo de edad
g_edad                      18-30       31-45       46-60         61+        TOTAL
ramo                                                                              
Accidentes Personales    5.248369    8.348395    8.731668    2.652247    24.980679
Autos                   55.457022   84.225512   84.571404   29.569569   253.823506
GMM                    139.278258  223.908688  264.218298  103.617078   731.022322
Vida                    79.667849  134.774641  166.301773   73.587860   454.332123
TOTAL                  279.651498  451.257236  523.823143  209.426754  1464.158631
                              Polizas por estado y ramo
ramo            estado  Accidentes Personales  Autos    GMM  Vida  TOTAL
0      Baja California                    321    990   1508   529   3348
1                 CDMX                    357   1000   1485   515   3357
2            Chihuahua                    360    972   1501   517   3350
3         

C:\Users\derec\AppData\Local\Temp\ipykernel_8140\1692146540.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_prima = pd.pivot_table(
C:\Users\derec\AppData\Local\Temp\ipykernel_8140\1692146540.py:11: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_estado = pd.pivot_table(


In [122]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 5: EXPORTAR
# ══════════════════════════════════════════════════════════════════════════════
# Excel con 5 hojas: Cartera_Limpia, Resumen_Ramo, Resumen_Agente,
#                    Pivot_Prima, Pivot_Zona
# Parquet: cartera_q1_2026_final.parquet
# Compara tamano CSV equivalente vs Parquet

# Tu codigo aqui:
with pd.ExcelWriter('datos/reporte_ejecutivo_Q1_2026.xlsx', engine='openpyxl') as writer:
    df_final.to_excel(writer, sheet_name='Cartera_Limpia', index=False)
    resumen_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    resumen_agente.to_excel(writer, sheet_name='Resumen_Agente', index=False)
    pivot_prima.to_excel(writer, sheet_name='Pivot_Prima')
    pivot_estado.to_excel(writer, sheet_name='Pivot_Zona')

kb_excel = os.path.getsize('datos/reporte_ejecutivo_Q1_2026.xlsx')/1024
print(f'Archivo Excel final: {kb_excel:.0f} KB con 5 hojas')

df_final.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)
kb_parquet = os.path.getsize('datos/cartera_q1_2026_final.parquet')/1024
print(f'CSV: {df_final.to_csv(index=False).encode().__len__()/1024:.0f} KB')
print(f'Parquet: {os.path.getsize("datos/cartera_q1_2026_final.parquet")/1024:.0f} KB')



Archivo Excel final: 10689 KB con 5 hojas
CSV: 14274 KB
Parquet: 2085 KB


In [ ]:
# ── Solucion resumida (descomenta solo si necesitas referencia) ──────────────

# FASE 1:
# df = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS, na_values=['N/D','N/A'])
# ramos_cat = pd.read_csv('datos/catalogo_ramos.csv')
# agentes_cat = pd.read_csv('datos/catalogo_agentes.csv')

# FASE 2:
# df = df.drop_duplicates()
# df['sexo'] = df['sexo'].str.strip().str.upper().map(MAPA_SEXO)
# for col in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
#     df[col] = pd.to_datetime(df[col], errors='coerce')
# df['fecha_nacimiento'] = pd.to_datetime(df['fecha_nacimiento'],format='%d/%m/%Y',errors='coerce')
# df['prima_neta'] = df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# df['codigo_postal'] = df['codigo_postal'].replace('N/D', np.nan)
# for col in ['ramo','plan','canal_venta','forma_pago','estado','sexo','tipo_vehiculo']:
#     if col in df.columns: df[col] = df[col].astype('category')

# FASE 3 (extracto):
# df = pd.merge(df, ramos_cat[['ramo','nombre_largo','tasa_base']], on='ramo', how='left')
# df['g_edad'] = pd.cut(df['edad'],bins=[0,30,45,60,100],labels=['18-30','31-45','46-60','61+'])
# df['prima_calc'] = df['suma_asegurada'] * df['tasa_base'] * 1.16
# from mi_modulo import clasificar_riesgo
# df['nivel_riesgo'] = df['prima_calc'].apply(lambda p: 'ALTO' if p>15000 else 'MEDIO' if p>6000 else 'BAJO')

# FASE 5:
# with pd.ExcelWriter('datos/reporte_ejecutivo_Q1_2026.xlsx', engine='openpyxl') as w:
#     df.to_excel(w,'Cartera_Limpia',index=False)
#     resumen_ramo.to_excel(w,'Resumen_Ramo',index=False)
#     resumen_agente.to_excel(w,'Resumen_Agente',index=False)
#     tabla_prima.to_excel(w,'Pivot_Prima')
#     tabla_zona.to_excel(w,'Pivot_Zona')
# df.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)
# print(f'CSV: {df.to_csv(index=False).encode().__len__()/1024:.0f} KB')
# print(f'Parquet: {os.path.getsize("datos/cartera_q1_2026_final.parquet")/1024:.0f} KB')

---
## Resumen: Lo que Aprendiste en la Sesion 8

| Duda | Herramienta | Aprendizaje clave |
|------|-------------|-------------------|
| 46 columnas | `usecols` + taxonomia | Clasificar antes de cargar — decision de negocio |
| Texto sucio | `str.strip/upper/map` | Normalizar ANTES del primer groupby |
| Fechas texto | `pd.to_datetime(errors='coerce')` | Nunca detener el pipeline por fechas invalidas |
| JSON anidado | `json_normalize(sep='_')` | Aplanar antes de analizar |
| 90k filas | `chunksize` + `category` | Medir memoria antes de decidir |
| Downcast riesgoso | Regla float32 < 100k MXN | float64 para montos grandes siempre |
| Polars | `pl.scan_csv().collect()` | Lazy evaluation = optimizacion automatica |

**T5 Pandas — COMPLETADO**

**Proxima sesion — Mie 6 mayo, 18:00 h:**
T6 Visualizacion — Matplotlib, Seaborn y Plotly.

```bash
git add sesion8_M1_notebook.ipynb
git commit -m "Sesion 8: pipeline completo datos reales - str datetime JSON Polars"
git push
```

---
*Diplomado ML en Seguros · FC UNAM · 2026*